<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/918_EPOv3_NodeTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Node Testing

Your **node testing layer is very solid** and shows the same engineering maturity as the utility tests. This is exactly the right next step in your testing pyramid. I'll walk through:


# 1. Overall Evaluation

This node test suite is **well structured and appropriate for orchestration systems**.

Your tests correctly focus on **node behavior**, not the graph runtime.

That means you are validating:

```text
node logic
state mutation
closure configuration
file IO
report generation
```

without needing to execute the full LangGraph pipeline.

That is exactly how most **LangGraph / Airflow / Prefect orchestration systems** are tested.

---

# 2. What You Did Extremely Well

## 1. Testing Nodes Independently

You correctly test each node:

```python
goal_node
planning_node
data_loading_node
portfolio_node
report_node
```

This mirrors the actual architecture:

```text
goal
↓
planning
↓
data loading
↓
portfolio intelligence
↓
report generation
```

Each node is validated independently.

This makes failures **easy to diagnose**.

---

# 2. Testing Closure Nodes Properly

This is very good:

```python
node = make_data_loading_node(config, tmp_path)
```

and

```python
node = make_report_node(config, tmp_path)
```

Closure nodes are notoriously tricky to test, and you handled them correctly by:

* constructing config
* injecting temp paths
* executing returned node

This ensures:

```text
configuration wiring works
paths resolve correctly
closures behave correctly
```

Excellent.

---

# 3. Using `tmp_path`

You use pytest’s built-in temp filesystem:

```python
tmp_path
```

This prevents tests from touching real files.

Example:

```python
(tmp_path / "experiment_runs.json").write_text(...)
```

That is exactly the correct way to test IO.

---

# 4. Good Edge Case Coverage

You test **failure and empty cases**, which many people skip.

Examples:

### Missing goal

```python
test_planning_node_requires_goal
```

### Missing files

```python
test_data_loading_node_missing_files_returns_zero_counts
```

### Empty snapshots

```python
test_portfolio_node_empty_data
```

These tests are extremely valuable.

---

# 5. Testing Report Output

This is one of the best parts:

```python
content = Path(out["report_file_path"]).read_text()
assert "One view" in content
assert "Portfolio status: healthy" in content
assert "Traceability" in content
```

You are validating **both behavior and output**.

This protects against accidental changes to the report format.

---

# 3. Coverage Analysis

Your node tests cover the **full orchestrator lifecycle**.

| Node                         | Tested |
| ---------------------------- | ------ |
| Goal node                    | ✔      |
| Planning node                | ✔      |
| Data loading                 | ✔      |
| Portfolio rollup integration | ✔      |
| Risk rollup integration      | ✔      |
| Trigger generation           | ✔      |
| Report generation            | ✔      |
| File writing                 | ✔      |
| Traceability metadata        | ✔      |

You are effectively testing **the entire orchestration pipeline minus LangGraph**.

That is ideal.

---

# 4. Two Small Improvements

These are **minor improvements**, not required.

---

# Improvement 1

## Test `run_id` generation format

You already test:

```python
assert out["run_id"].startswith("epo_v3_")
```

But you could also verify timestamp structure.

Example:

```python
def test_goal_node_run_id_format():
    state = {"errors": []}
    out = goal_node(state)

    assert out["run_id"].startswith("epo_v3_")
    assert len(out["run_id"]) > 10
```

This prevents accidental format changes.

---

# Improvement 2

## Verify Health Score Propagation

Your portfolio node computes:

```text
portfolio_health_score
```

You check that it exists, but not its value.

Add:

```python
assert isinstance(out["portfolio_rollup"]["portfolio_health_score"], int)
```

This protects against accidental `None` or string values.

---

# 5. One Very Powerful Advanced Test

Large companies often add **state contract tests**.

Example:

```python
def test_portfolio_node_state_contract():
    config = EPOv3OrchestratorConfig()
    node = make_portfolio_node(config)

    state = {
        "errors": [],
        "experiment_portfolio_snapshots": [],
        "experiment_risk_events": [],
    }

    out = node(state)

    required_keys = [
        "portfolio_rollup",
        "risk_rollup",
        "executive_triggers",
        "errors",
    ]

    for key in required_keys:
        assert key in out
```

This protects the **state interface between nodes**.

Which is critical in orchestration systems.

---

# 6. Your Testing Pyramid is Now Excellent

You now have three levels:

### Level 1 — Utility Tests

Validate core logic.

```text
data loading
rollups
risk logic
triggers
report builder
```

---

### Level 2 — Node Tests

Validate orchestration logic.

```text
node state behavior
closures
file IO
report writing
```

---

### Level 3 — (Next Step)

You could optionally add **one integration test**:

```text
run_epo_v3.py
```

That confirms the graph executes end-to-end.

But even without it, your coverage is already strong.

---

# Final Assessment

Your node testing is:

```text
clean
deterministic
well isolated
IO safe
well structured
```

You are testing the **exact boundaries where orchestration systems fail**:

* node state mutation
* closure wiring
* filesystem output
* report generation

That is excellent engineering practice.



In [ ]:
"""
Node tests for EPOv3: goal, planning, data_loading (closure), portfolio (closure), report (closure).
Use mock state; temp dirs for data and report output.
"""
import json
import sys
from pathlib import Path

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pytest

from config import EPOv3OrchestratorConfig
from agents.epo_v3.orchestrator.nodes import (
    goal_node,
    planning_node,
    make_data_loading_node,
    make_portfolio_node,
    make_report_node,
)


# ---- Goal node ----


def test_goal_node_sets_goal_and_run_metadata():
    state = {"errors": []}
    out = goal_node(state)
    assert "goal" in out
    assert out["goal"]["objective"]
    assert out["goal"]["focus_areas"] == [
        "experiment_effectiveness", "design_quality", "risk_exposure", "learning_velocity"
    ]
    assert "run_id" in out
    assert out["run_id"].startswith("epo_v3_")
    assert "run_started_at" in out
    assert out["errors"] == []


def test_goal_node_preserves_run_id_if_given():
    state = {"errors": [], "run_id": "custom_123"}
    out = goal_node(state)
    assert out["run_id"] == "custom_123"


# ---- Planning node ----


def test_planning_node_requires_goal():
    state = {"errors": []}
    out = planning_node(state)
    assert "planning_node: goal is required" in out["errors"]


def test_planning_node_returns_plan():
    state = {"goal": {"objective": "Analyze portfolio"}, "errors": []}
    out = planning_node(state)
    assert "plan" in out
    assert len(out["plan"]) == 3
    names = [s["name"] for s in out["plan"]]
    assert "data_loading" in names
    assert "portfolio" in names
    assert "report" in names


# ---- Data loading node (closure) ----


def test_data_loading_node_loads_from_tmp_dir(tmp_path):
    (tmp_path / "experiment_runs.json").write_text(json.dumps([{"experiment_id": "E1"}]))
    (tmp_path / "experiment_portfolio_snapshots.json").write_text("[]")
    (tmp_path / "experiment_risk_events.json").write_text("[]")
    (tmp_path / "experiment_task_execution_logs.json").write_text("[]")
    config = EPOv3OrchestratorConfig()
    config.data_dir = str(tmp_path)
    node = make_data_loading_node(config, tmp_path)
    state = {"errors": []}
    out = node(state)
    assert out["runs_count"] == 1
    assert out["experiments_count"] == 1
    assert out["snapshots_count"] == 0
    assert "data_snapshot_loaded_at" in out
    assert out["errors"] == []


def test_data_loading_node_missing_files_returns_zero_counts(tmp_path):
    """Missing JSON files yield empty lists and zero counts, no exception."""
    config = EPOv3OrchestratorConfig()
    config.data_dir = str(tmp_path)
    node = make_data_loading_node(config, tmp_path)
    state = {"errors": []}
    out = node(state)
    assert out["runs_count"] == 0
    assert out["experiments_count"] == 0
    assert out["errors"] == []


# ---- Portfolio node (closure) ----


def test_portfolio_node_builds_rollups_and_triggers():
    config = EPOv3OrchestratorConfig()
    node = make_portfolio_node(config)
    state = {
        "errors": [],
        "experiment_portfolio_snapshots": [
            {"snapshot_date": "2024-10-01", "experiments_at_risk": 1, "high_risk_experiments": 1, "learning_velocity_last_30d": 2},
        ],
        "experiment_risk_events": [
            {"severity": "critical", "risk_type": "compliance_risk", "status": "blocked"},
        ],
    }
    out = node(state)
    assert "portfolio_rollup" in out
    assert out["portfolio_rollup"]["experiments_at_risk"] == 1
    assert "portfolio_health_score" in out["portfolio_rollup"]
    assert "risk_rollup" in out
    assert out["risk_rollup"]["critical_count"] == 1
    assert "executive_triggers" in out
    assert len(out["executive_triggers"]) >= 1
    assert out["errors"] == []


def test_portfolio_node_empty_data():
    config = EPOv3OrchestratorConfig()
    node = make_portfolio_node(config)
    state = {"errors": [], "experiment_portfolio_snapshots": [], "experiment_risk_events": []}
    out = node(state)
    assert out["portfolio_rollup"]["active_experiments"] == 0
    assert out["risk_rollup"]["total_events"] == 0
    assert out["errors"] == []


# ---- Report node (closure) ----


def test_report_node_writes_file_and_returns_path(tmp_path):
    config = EPOv3OrchestratorConfig()
    config.reports_dir = str(tmp_path)
    node = make_report_node(config, tmp_path)
    state = {
        "errors": [],
        "portfolio_rollup": {
            "portfolio_status": "healthy",
            "experiments_at_risk": 0,
            "high_risk_experiments": 0,
            "learning_velocity_last_30d": 3,
            "portfolio_health_score": 90,
            "active_experiments": 2,
            "delta_experiments_at_risk": None,
            "delta_learning_velocity": None,
            "estimated_annual_value_in_flight_usd": 100000,
        },
        "risk_rollup": {"open_count": 0, "by_severity_open": {}, "by_risk_type": {}, "dominant_risk_type": "none", "total_events": 0},
        "executive_triggers": [],
        "runs_count": 5,
        "snapshots_count": 2,
        "risk_events_count": 0,
        "task_logs_count": 5,
        "experiments_count": 2,
        "run_id": "test_run",
        "run_started_at": "2024-01-01T12:00:00Z",
    }
    out = node(state)
    assert out["errors"] == []
    assert "report_file_path" in out
    assert out["report_file_path"].startswith(str(tmp_path))
    assert Path(out["report_file_path"]).exists()
    content = Path(out["report_file_path"]).read_text()
    assert "One view" in content
    assert "Portfolio status: healthy" in content
    assert "Traceability" in content
